In [0]:
-- SELECT * FROM bronze_inventory;

CREATE OR REPLACE TABLE silver_sales AS
SELECT
  `Order Id`                    AS order_id,
  `Order Item Id`               AS order_item_id,
  `Product Id`                  AS product_id,
  `Product Name`                AS product_name,
  `Product Category Id`         AS product_category_id,
  `Category Name`               AS category_name,
  `Department Id`               AS department_id,
  `Department Name`             AS department_name,

  -- Standardize customer and market attributes for downstream analysis
  `Customer Id`                 AS customer_id,
  `Customer Segment`            AS customer_segment,
  `Customer City`               AS customer_city,
  `Customer Country`            AS customer_country,
  `Market`                      AS market,
  `Order Region`                AS order_region,

  -- Parse order and shipping timestamps from multiple source formats
  COALESCE(
    TRY_TO_DATE(`order date (DateOrders)`, 'M/d/yyyy H:mm'),
    TRY_TO_DATE(`order date (DateOrders)`, 'M/d/yy H:mm')
  ) AS order_date,
  COALESCE(
    TRY_TO_DATE(`shipping date (DateOrders)`, 'M/d/yyyy H:mm'),
    TRY_TO_DATE(`shipping date (DateOrders)`, 'M/d/yy H:mm')
  ) AS ship_date,

  -- Derive delivery performance metrics
  `Shipping Mode`               AS shipping_mode,
  `Delivery Status`             AS delivery_status,
  `Days for shipment (scheduled)` AS scheduled_ship_days,
  `Days for shipping (real)`    AS actual_ship_days,
  (`Days for shipping (real)` - `Days for shipment (scheduled)`) AS delivery_delay,
  CASE WHEN `Delivery Status` = 'Late delivery' THEN 1 ELSE 0 END AS is_late_delivery,

  -- Derive sales, discount, and profitability metrics
  `Sales`                       AS sales,
  `Sales per customer`          AS sales_per_customer,
  `Order Item Quantity`         AS order_qty,
  `Order Item Product Price`    AS unit_price,
  `Order Item Discount`         AS discount_amount,
  `Order Item Discount Rate`    AS discount_rate,
  `Order Item Total`            AS order_total,
  `Order Profit Per Order`      AS profit_per_order,
  `Order Item Profit Ratio`     AS profit_ratio,
  `Benefit per order`           AS benefit_per_order,
  CASE WHEN `Order Profit Per Order` > 0 THEN 1 ELSE 0 END AS is_profitable,

  -- Preserve the final order status and payment type fields
  `Order Status`                AS order_status,
  `Type`                        AS payment_type

FROM bronze_sales
WHERE `Order Item Id` IS NOT NULL
  AND `Product Id` IS NOT NULL
  AND `Order Item Quantity` IS NOT NULL;



-- Rebuild the inventory silver table with normalized stock metrics

CREATE OR REPLACE TABLE silver_inventory AS
SELECT
  `product id`      AS product_id,
  `product name`    AS product_name,
  `current stock`   AS current_stock,
  `reorder point`   AS reorder_point,
  `safety stock`    AS safety_stock,
  `avg lead time`   AS avg_lead_time,
  `max lead time`   AS max_lead_time,
  `avg order qty`   AS avg_order_qty,
  `max order qty`   AS max_order_qty,
  `mod`             AS min_order_qty,
  `order-now`       AS order_now_flag,

  -- Derive inventory risk and replenishment features
  CASE WHEN `order-now` = 'orange' THEN 1 ELSE 0 END AS stockout_risk,
  (`current stock` - `safety stock`)                  AS stock_buffer,
  CASE 
    WHEN `current stock` <= `safety stock` THEN 'Critical'
    WHEN `current stock` <= `reorder point` THEN 'Low'
    ELSE 'Healthy'
  END AS stock_status

FROM bronze_inventory
WHERE `product id` IS NOT NULL;


CREATE OR REPLACE TABLE silver_supply_chain AS
SELECT
  s.*,

  -- Bring inventory attributes into the sales grain
  i.current_stock,
  i.reorder_point,
  i.safety_stock,
  i.avg_lead_time,
  i.max_lead_time,
  i.avg_order_qty,
  i.max_order_qty,
  i.min_order_qty,
  i.stockout_risk,
  i.stock_buffer,
  i.stock_status,
  i.order_now_flag

FROM silver_sales s
LEFT JOIN silver_inventory i
  ON s.product_id = i.product_id;


-- Validate row counts across the silver tables

SELECT 'silver_sales' AS table_name, COUNT(*) AS row_count FROM silver_sales
UNION ALL
SELECT 'silver_inventory', COUNT(*) FROM silver_inventory
UNION ALL
SELECT 'silver_supply_chain', COUNT(*) FROM silver_supply_chain;



-- Validate join coverage and timestamp completeness

SELECT
  COUNT(*)                          AS total_rows,
  COUNT(current_stock)              AS matched_with_inventory,
  COUNT(*) - COUNT(current_stock)   AS unmatched_rows,
  COUNT(order_date)                 AS valid_order_dates,
  COUNT(*) - COUNT(order_date)      AS null_dates,
  SUM(is_late_delivery)             AS total_late_deliveries,
  ROUND(AVG(delivery_delay), 2)     AS avg_delivery_delay
FROM silver_supply_chain;



-- Rebuild silver_sales with corrected delivery-status mappings

CREATE OR REPLACE TABLE silver_sales AS
SELECT
  `Order Id`                      AS order_id,
  `Order Item Id`                 AS order_item_id,
  `Product Id`                    AS product_id,
  `Product Name`                  AS product_name,
  `Product Category Id`           AS product_category_id,
  `Category Name`                 AS category_name,
  `Department Id`                 AS department_id,
  `Department Name`               AS department_name,

  -- Standardize customer and market attributes for downstream analysis
  `Customer Id`                   AS customer_id,
  `Customer Segment`              AS customer_segment,
  `Customer City`                 AS customer_city,
  `Customer Country`              AS customer_country,
  `Market`                        AS market,
  `Order Region`                  AS order_region,

  -- Parse order and shipping timestamps from multiple source formats
  COALESCE(
    TRY_TO_DATE(`order date (DateOrders)`, 'M/d/yyyy H:mm'),
    TRY_TO_DATE(`order date (DateOrders)`, 'M/d/yy H:mm')
  ) AS order_date,

  COALESCE(
    TRY_TO_DATE(`shipping date (DateOrders)`, 'M/d/yyyy H:mm'),
    TRY_TO_DATE(`shipping date (DateOrders)`, 'M/d/yy H:mm')
  ) AS ship_date,

  -- Derive delivery performance metrics
  `Shipping Mode`                 AS shipping_mode,
  `Delivery Status`               AS delivery_status,
  `Days for shipment (scheduled)` AS scheduled_ship_days,
  `Days for shipping (real)`      AS actual_ship_days,
  (`Days for shipping (real)` - `Days for shipment (scheduled)`) AS delivery_delay,

  -- Correct delivery-status flags to match the source values
  CASE 
    WHEN `Delivery Status` = 'Late'     THEN 1 
    ELSE 0 
  END AS is_late_delivery,

  CASE
    WHEN `Delivery Status` = 'Canceled' THEN 1
    ELSE 0
  END AS is_canceled,

  CASE
    WHEN `Delivery Status` = 'Advance'  THEN 1
    ELSE 0
  END AS is_early_delivery,

  -- Derive sales, discount, and profitability metrics
  `Sales`                         AS sales,
  `Sales per customer`            AS sales_per_customer,
  `Order Item Quantity`           AS order_qty,
  `Order Item Product Price`      AS unit_price,
  `Order Item Discount`           AS discount_amount,
  `Order Item Discount Rate`      AS discount_rate,
  `Order Item Total`              AS order_total,
  `Order Profit Per Order`        AS profit_per_order,
  `Order Item Profit Ratio`       AS profit_ratio,
  `Benefit per order`             AS benefit_per_order,
  CASE 
    WHEN `Order Profit Per Order` > 0 THEN 1 
    ELSE 0 
  END AS is_profitable,

  -- Preserve the final order status and payment type fields
  `Order Status`                  AS order_status,
  `Type`                          AS payment_type

FROM bronze_sales
WHERE `Order Item Id`       IS NOT NULL
  AND `Product Id`          IS NOT NULL
  AND `Order Item Quantity` IS NOT NULL;



-- Rebuild the supply-chain join using the corrected silver_sales table
CREATE OR REPLACE TABLE silver_supply_chain AS
SELECT
  s.*,
  i.current_stock,
  i.reorder_point,
  i.safety_stock,
  i.avg_lead_time,
  i.max_lead_time,
  i.avg_order_qty,
  i.max_order_qty,
  i.min_order_qty,
  i.stockout_risk,
  i.stock_buffer,
  i.stock_status,
  i.order_now_flag
FROM silver_sales s
LEFT JOIN silver_inventory i
  ON s.product_id = i.product_id;



-- Validate delivery metrics after the rebuild
SELECT
  COUNT(*)                        AS total_rows,
  COUNT(order_date)               AS valid_order_dates,
  COUNT(*) - COUNT(order_date)    AS null_dates,
  SUM(is_late_delivery)           AS total_late_deliveries,
  SUM(is_canceled)                AS total_canceled,
  SUM(is_early_delivery)          AS total_early_deliveries,
  ROUND(AVG(delivery_delay), 2)   AS avg_delivery_delay
FROM silver_supply_chain;


-- Inspect the bronze sources used for the store inventory silver table
-- Bronze source: supply-chain snapshot
-- Bronze source: store inventory snapshot

SELECT * FROM bronze_supply_chain LIMIT 5;
SELECT * FROM bronze_store_inventory LIMIT 5;

-- Create the silver store inventory table with standardized fields

CREATE OR REPLACE TABLE silver_store_inventory AS
SELECT
  CAST(Date AS DATE)          AS inventory_date,
  `Store ID`                  AS store_id,
  `Product ID`                AS product_id,
  Category                    AS category,
  Region                      AS region,
  `Inventory Level`           AS inventory_level,
  `Units Sold`                AS units_sold,
  `Units Ordered`             AS units_ordered,

  -- Derive stock movement and inventory-health features
  (`Inventory Level` - `Units Sold`)    AS stock_after_sales,
  (`Units Ordered` - `Units Sold`)      AS order_fulfillment_gap,
  CASE
    WHEN `Units Sold` > `Inventory Level` THEN 1
    ELSE 0
  END AS is_stockout,
  CASE
    WHEN `Inventory Level` < 50  THEN 'Critical'
    WHEN `Inventory Level` < 150 THEN 'Low'
    WHEN `Inventory Level` < 300 THEN 'Medium'
    ELSE 'High'
  END AS inventory_health

FROM bronze_store_inventory
WHERE `Product ID` IS NOT NULL
  AND Date IS NOT NULL;


-- Build the final supply-chain feature table from the bronze source

CREATE OR REPLACE TABLE silver_supply_chain_data AS
SELECT
  `Product type`              AS product_type,
  `SKU`                       AS sku,
  CAST(`Price` AS DOUBLE)     AS price,
  `Availability`              AS availability,
  `Number of products sold`   AS products_sold,
  `Revenue generated`         AS revenue,
  `Customer demographics`     AS customer_segment,
  `Stock levels`              AS stock_level,
  `Lead times`                AS lead_time,
  `Order quantities`          AS order_quantity,
  `Shipping times`            AS shipping_time,
  `Shipping carriers`         AS shipping_carrier,
  `Shipping costs`            AS shipping_cost,
  `Supplier name`             AS supplier_name,
  `Location`                  AS supplier_location,
  `Lead time`                 AS supplier_lead_time,
  `Production volumes`        AS production_volume,
  `Manufacturing lead time`   AS manufacturing_lead_time,
  `Manufacturing costs`       AS manufacturing_cost,
  `Inspection results`        AS inspection_result,
  `Defect rates`              AS defect_rate,
  `Transportation modes`      AS transport_mode,
  `Routes`                    AS route,
  `Costs`                     AS total_cost,

  -- Derive operational and supplier risk features
  ROUND(`Revenue generated` / NULLIF(`Number of products sold`, 0), 2) 
                              AS revenue_per_unit,
  ROUND(`Shipping costs` / NULLIF(`Order quantities`, 0), 2)           
                              AS shipping_cost_per_unit,
  CASE
    WHEN `Inspection results` = 'Fail' THEN 1
    ELSE 0
  END                         AS is_defective,
  CASE
    WHEN `Defect rates` > 3.0 THEN 'High Risk'
    WHEN `Defect rates` > 1.5 THEN 'Medium Risk'
    ELSE 'Low Risk'
  END                         AS defect_risk_level,
  CASE
    WHEN `Stock levels` < 10  THEN 'Critical'
    WHEN `Stock levels` < 30  THEN 'Low'
    WHEN `Stock levels` < 60  THEN 'Medium'
    ELSE 'High'
  END                         AS stock_health

FROM bronze_supply_chain
WHERE `SKU` IS NOT NULL;



SELECT 'silver_store_inventory'   AS table_name, COUNT(*) AS rows FROM silver_store_inventory
UNION ALL
SELECT 'silver_supply_chain_data' AS table_name, COUNT(*) AS rows FROM silver_supply_chain_data;


SHOW TABLES;